# Trial Eligibility

**Assess number of patients in the aNSCLC analytic cohort meeting KEYNOTE-048 ECOG and lab eligibility.**

In [1]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorHeadNeck

## Import dfs

In [2]:
index_df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
index_df.shape

(1854, 3)

In [4]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [5]:
features_df.shape

(1724, 167)

In [6]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [7]:
surv_pred_df.shape

(1471, 2)

In [8]:
main_df = pd.merge(features_df, index_df, on = 'PatientID', how = 'left')

In [9]:
main_df = pd.merge(main_df, surv_pred_df, on = 'PatientID', how = 'left')

In [10]:
main_df.shape

(1724, 170)

In [11]:
main_df['StartDate'] = pd.to_datetime(main_df['StartDate'])

In [12]:
main_df['treatment_year'] = main_df['StartDate'].dt.year

In [13]:
main_df = main_df.query('treatment_year <= 2022')

In [14]:
main_df.shape

(1471, 171)

In [15]:
main_df = main_df[['PatientID', 'LineName', 'StartDate', 'psurv_180_calibrated']]

In [16]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated
1,FEBCC4F65CC4C,pembro,2016-12-16,0.979187
2,F2C55497C35F5,pembro_platinum,2021-10-18,0.046632
3,F51465980052A,pembro_platinum,2019-05-10,0.863684


## ECOG and Labs

In [17]:
# Initialize class 
processor = DataProcessorHeadNeck()

In [18]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-11-26 21:26:03,768 - INFO - Successfully read Lab.csv file with shape: (9064402, 17) and unique PatientIDs: 10989
2026-11-26 21:26:05,748 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (1474009, 18) and unique PatientIDs: 1459
2026-11-26 21:26:09,058 - INFO - Successfully processed Lab.csv file with final shape: (1471, 81) and unique PatientIDs: 1471


In [19]:
labs_df = labs_df[['PatientID', 'anc', 'hemoglobin', 'platelet', 'creatinine', 'total_bilirubin', 'ast', 'alt', 'alp']]

In [20]:
labs_df.shape

(1471, 9)

In [21]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-11-26 21:26:09,173 - INFO - Successfully read ECOG.csv file with shape: (199312, 4) and unique PatientIDs: 8957
2026-11-26 21:26:09,208 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (35689, 5) and unique PatientIDs: 1307
2026-11-26 21:26:09,236 - INFO - Successfully processed ECOG.csv file with final shape: (1471, 3) and unique PatientIDs: 1471


In [22]:
ecog_df = ecog_df[['PatientID', 'ecog_index']]

In [23]:
ecog_df.shape

(1471, 2)

In [24]:
main_df = pd.merge(main_df, labs_df, on = 'PatientID', how = 'left')

In [25]:
main_df = pd.merge(main_df, ecog_df, on = 'PatientID', how = 'left')

In [26]:
main_df.shape

(1471, 13)

In [27]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated,anc,hemoglobin,platelet,creatinine,total_bilirubin,ast,alt,alp,ecog_index
0,FEBCC4F65CC4C,pembro,2016-12-16,0.979187,4.80,14.9,227.0,1.2,0.4,19.0,14.0,58.0,1
1,F2C55497C35F5,pembro_platinum,2021-10-18,0.046632,13.73,12.5,428.0,0.7,1.0,46.0,37.0,648.0,2
2,F51465980052A,pembro_platinum,2019-05-10,0.863684,10.24,11.5,162.0,0.9,0.4,13.0,11.0,40.0,1


## Risk stratification

In [28]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate = float(f.read())

In [29]:
main_df['high_risk'] = np.where(main_df['psurv_180_calibrated'] < crossover_survival_estimate, 1, 0)

In [30]:
main_df.high_risk.value_counts()

high_risk
0    1083
1     388
Name: count, dtype: int64

## ECOG Eligibility

In [31]:
main_df['ecog_eligible'] = np.where((main_df['ecog_index'] <= 2), 1, 0)

In [32]:
main_df['ecog_ineligible'] = np.where(main_df['ecog_index'] > 2, 1, 0)

In [33]:
main_df['ecog_indeterminate'] = np.where(main_df['ecog_index'].isna(), 1, 0)

In [34]:
main_df.shape[0]

1471

In [35]:
main_df.ecog_eligible.value_counts()[1]+main_df.ecog_ineligible.value_counts()[1]+main_df.ecog_indeterminate.value_counts()[1]

np.int64(1471)

## Lab eligibility

In [36]:
main_df['lab_eligible'] = np.where(
    (main_df['anc'] >= 1.5) &
    (main_df['platelet'] >= 100) &
    (main_df['hemoglobin'] >= 8) &
    (main_df['creatinine'] <= 1.8) & 
    (main_df['total_bilirubin'] <= 1.8) &
    (main_df['ast'] <= 100) &
    (main_df['alt'] <= 100) &
    (main_df['alp'] <= 300), 1, 0)

In [37]:
main_df['lab_ineligible'] = np.where(
    (main_df['anc'] < 1.5) |
    (main_df['platelet'] < 100) |
    (main_df['hemoglobin'] < 8) |
    (main_df['creatinine'] > 1.8) | 
    (main_df['total_bilirubin'] > 1.8) |
    (main_df['ast'] > 100) |
    (main_df['alt'] > 100) |
    (main_df['alp'] > 300), 1, 0)

In [38]:
main_df['lab_indeterminate'] = np.where((main_df['lab_eligible'] == 0) & (main_df['lab_ineligible'] == 0), 1, 0)

In [39]:
main_df.lab_eligible.value_counts()[1]+main_df.lab_ineligible.value_counts()[1]+main_df.lab_indeterminate.value_counts()[1]

np.int64(1471)

## Overal eligibility

In [40]:
main_df['eligible'] = np.where((main_df['ecog_eligible'] == 1) & (main_df['lab_eligible'] == 1), 1, 0)

In [41]:
main_df['ineligible'] = np.where((main_df['ecog_ineligible'] == 1) | (main_df['lab_ineligible'] == 1), 1, 0)

In [42]:
main_df['indeterminate'] = np.where((main_df['eligible'] == 0) & (main_df['ineligible'] == 0), 1, 0)

In [43]:
print('Overall cohort')
print(f'Eligible: count {main_df.eligible.value_counts()[1]}, percentage {round(main_df.eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.ineligible.value_counts()[1]}, percentage {round(main_df.ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.indeterminate.value_counts()[1]}, percentage {round(main_df.indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('High-risk')
print(f'Eligible: count {main_df.query('high_risk == 1').eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 1').ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 1').indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('Low-risk')
print(f'Eligible: count {main_df.query('high_risk == 0').eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 0').ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 0').indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').indeterminate.value_counts(normalize = True)[1]*100, 2)}')

Overall cohort
Eligible: count 758, percentage 51.53
Ineligible: count 140, percentage 9.52
Indeterminate: count 573, percentage 38.95

High-risk
Eligible: count 178, percentage 45.88
Ineligible: count 87, percentage 22.42
Indeterminate: count 123, percentage 31.7

Low-risk
Eligible: count 580, percentage 53.55
Ineligible: count 53, percentage 4.89
Indeterminate: count 450, percentage 41.55
